# Poiseuilleov tok: predvidi → izračunaj → provjeri

Za potpuno razvijen laminarni tok Newtonova fluida u kružnoj cijevi profil je paraboličan. Notebook razdvaja dvije vrste pitanja:

- **numeričku pogrešku** integracije profila po mreži;
- **ulaznu nesigurnost** protoka zbog mjerenja polumjera, tlaka, viskoznosti i duljine.

## Predvidi

1. Hoće li trapezna integracija konvergirati kada se broj intervala udvostručuje?
2. Može li finija mreža ukloniti pogrešku primjene laminarnog modela na turbulentan tok?
3. Zašto mala relativna nesigurnost polumjera može snažno utjecati na protok?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

radius, delta_p, mu, length = 0.025, 60.0, 1.0e-3, 3.0
umax = delta_p*radius**2/(4*mu*length)
q_exact = np.pi*radius**4*delta_p/(8*mu*length)

def q_trapezoid(n):
    r = np.linspace(0, radius, n+1)
    u = umax*(1-(r/radius)**2)
    return np.trapezoid(2*np.pi*r*u, r)

levels = np.array([8, 16, 32, 64, 128])
q_num = np.array([q_trapezoid(int(n)) for n in levels])
errors = np.abs(q_num-q_exact)
orders = np.log(errors[:-1]/errors[1:])/np.log(2)
print(" n       Q_num (m3/s)       |pogreška|       opaženi red")
for i, n in enumerate(levels):
    order = "-" if i == 0 else f"{orders[i-1]:.3f}"
    print(f"{n:3d}   {q_num[i]:.9f}     {errors[i]:.3e}      {order}")


## Izračunaj: propagacija ulazne nesigurnosti

Za Poiseuilleov zakon \(Q\propto R^4\Delta p/(\mu L)\). Linearizirana relativna standardna nesigurnost zato sadrži član \(4u_R/R\). Rezultat uspoređujemo s Monte Carlo uzorkovanjem.


In [ ]:
sigma_R, sigma_dp, sigma_mu, sigma_L = 0.00010, 0.60, 1.0e-5, 0.003
relative_linear = np.sqrt(
    (4*sigma_R/radius)**2 + (sigma_dp/delta_p)**2 +
    (sigma_mu/mu)**2 + (sigma_L/length)**2
)
u_q_linear = q_exact*relative_linear

rng = np.random.default_rng(20260803)
n_samples = 40_000
R_mc = rng.normal(radius, sigma_R, n_samples)
dp_mc = rng.normal(delta_p, sigma_dp, n_samples)
mu_mc = rng.normal(mu, sigma_mu, n_samples)
L_mc = rng.normal(length, sigma_L, n_samples)
q_mc = np.pi*R_mc**4*dp_mc/(8*mu_mc*L_mc)
u_q_mc = np.std(q_mc, ddof=1)

contribution_radius = (4*sigma_R/radius)**2/relative_linear**2
print(f"u_Q linearno = {u_q_linear:.3e} m3/s; Monte Carlo = {u_q_mc:.3e} m3/s")
print(f"Polumjer nosi {100*contribution_radius:.1f} % linearizirane varijance.")


## Provjeri

Konvergencija provjerava red numeričke metode. Neovisno provjeravamo analitičku vezu \(Q=A\,u_{max}/2\) i slaganje dvaju postupaka propagacije nesigurnosti.


In [ ]:
assert np.all(errors[1:] < errors[:-1])
assert np.allclose(orders[-3:], 2.0, atol=0.02)
assert np.isclose(q_exact, np.pi*radius**2*umax/2, rtol=1e-14)
assert abs(u_q_mc/u_q_linear-1) < 0.05
assert contribution_radius > 0.5

r = np.linspace(0, radius, 250)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
axes[0].plot(1e3*r, umax*(1-(r/radius)**2), color="#256d85")
axes[0].set(xlabel="r (mm)", ylabel="u (m/s)", title="Analitički profil")
axes[1].loglog(levels, errors, "o-", color="#b43c35")
axes[1].set(xlabel="broj intervala", ylabel="$|Q_h-Q|$", title="Mrežna konvergencija")
axes[2].hist(1e3*q_mc, bins=50, color="#7cb5d6", edgecolor="white")
axes[2].set(xlabel="Q (L/s)", ylabel="broj uzoraka", title="Ulazna nesigurnost")
for ax in axes: ax.grid(True, ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Protumači

Udvostručenje mreže smanjuje diskretizacijsku pogrešku približno četiri puta, ali ne mijenja valjanost pretpostavke potpuno razvijenog laminarnog toka. Za usporedbu s mjerenjem treba zajednički prikazati numeričku, ulaznu, eksperimentalnu i modelsku nesigurnost.
